# Kiểm chứng Model 2 bằng dữ liệu THẬT (scene `hcm0031`, Round 1 CŨ đã bị huỷ)

**Notebook này CHỈ để kiểm chứng bằng số liệu thật (Score PSNR/SSIM/LPIPS) — KHÔNG phải
1 phần của luồng nộp bài Round 2.** Scene `hcm0031` thuộc dataset Round 1 cũ
(`Dataset/VAI_NVS_DATA/phase1/public_set/`) mà BTC đã huỷ và thay bằng dataset Round 2
mới (7 scene khác hẳn — xem `docs/00_MASTER_PLAN.md`). Lý do dùng scene này để kiểm
chứng: đây là dataset DUY NHẤT đang có sẵn ảnh GT test THẬT (Round 2 không cấp GT test
cho bất kỳ scene nào) — cho phép tính Score khách quan thật, đúng công thức BTC, để trả
lời câu hỏi "Model 2 có thực sự làm ảnh tốt hơn không, tăng bao nhiêu điểm".

Quy trình:
1. Nạp checkpoint Model 1 đã train sẵn (`gs_model/`, 30000 iteration, 4.9 triệu Gaussian).
2. Render toàn bộ 50 pose TEST → tính Score **TRƯỚC** khi sửa (so với ảnh GT test thật).
3. Render toàn bộ 200 pose TRAIN → làm cặp (render, GT) để train Model 2.
4. Train Model 2 (mạng sửa lỗi pixel, nhánh gate tự học vùng cần sửa).
5. Áp Model 2 lên render TEST → tính Score **SAU** khi sửa → so sánh.

Cần chuẩn bị 2 link Google Drive (điền ở Bước 4):
- **Checkpoint**: nén `Round1/gs_model/` thành zip, upload Drive, chia sẻ "Anyone with
  the link" (~1.2GB).
- **Dataset**: nén `Dataset/VAI_NVS_DATA/phase1/public_set/hcm0031/` (train/ + test/,
  ~280MB) thành zip, upload Drive tương tự — **BẮT BUỘC có cả `test/images/`** (ảnh GT
  test thật) để tính được Score, không chỉ `test/test_poses.csv`.


## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit(
        "KHÔNG có GPU khả dụng. Vào Settings -> Accelerator -> GPU T4 x2/P100 -> Save, "
        "chạy lại từ đầu. Render 3DGS cần CUDA."
    )
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip install -q "pycolmap>=3.10" "scikit-image>=0.19" lpips plyfile tqdm "gdown>=6,<7"

## Bước 2 — Clone + build 3D Gaussian Splatting

Cùng commit pin đã dùng cho toàn bộ pipeline (`54c035f7...`) — checkpoint này train
bằng bản CŨ hơn của `gaussian-splatting` (trước khi repo có `--antialiasing`), nhưng
`cfg_args`/log của notebook train gốc xác nhận KHÔNG bật antialiasing/depth/exposure
nào đặc biệt (`train_test_exp=False`, baseline thuần) nên render lại bằng bản pin hiện
tại vẫn cho kết quả nhất quán (không lệch cấu hình).


In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])


## Bước 3 — Lấy code pipeline (nhánh `feature/nn-image-corrector`)

Cần `pipeline/common/corrector_model.py` (Model 2) + `pipeline/common/poses.py` +
`pipeline/scripts/08_build_corrector_dataset.py`/`09_train_corrector.py`/
`10_apply_corrector.py`/`04_eval_metrics.py` — notebook này IMPORT các file đó làm THƯ
VIỆN (`importlib`, không chạy `main()` của chúng qua `!python ...`) để TÁI SỬ DỤNG đúng
logic đã test, mà KHÔNG cần đăng ký scene `hcm0031` vào `pipeline/common/scenes.py`
(scene này không thuộc Round 2, cố tình không đăng ký vào registry chính để tránh nhầm
lẫn với luồng nộp bài thật).


In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound.git"
GIT_BRANCH = "feature/nn-image-corrector"

GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public).")

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

import os, shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None:
    raise SystemExit(f"Không tìm thấy thư mục pipeline/ trong {CLONE_ROOT} — kiểm tra GIT_BRANCH.")

target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("pipeline ->", found.resolve())

for _rel in ("common/corrector_model.py", "scripts/08_build_corrector_dataset.py",
             "scripts/09_train_corrector.py", "scripts/10_apply_corrector.py",
             "scripts/04_eval_metrics.py"):
    if not (target / _rel).exists():
        raise SystemExit(f"Thiếu {target / _rel} — GIT_BRANCH='{GIT_BRANCH}' có đúng nhánh Model 2 không?")
print("Đủ file cần thiết.")

!git -C /kaggle/working/_repo_clone log --oneline -1
print(f"-> Commit vừa clone (đối chiếu với https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound/"
      f"commits/{GIT_BRANCH} nếu nghi ngờ code cũ).")

# Kiểm tra TRỰC TIẾP fix bug gate collapse (tail_gate.bias khởi tạo dương) có mặt trong
# bản code vừa clone hay không — bằng chứng CỤ THỂ hơn nhiều so với chỉ tin commit hash
# (phòng trường hợp lỡ trỏ nhánh/commit không có fix).
_corrector_src = (target / "common" / "corrector_model.py").read_text()
_has_gate_fix = "constant_(self.tail_gate.bias" in _corrector_src
print(f"Fix gate collapse (tail_gate.bias khởi tạo dương) có trong code vừa tải: {_has_gate_fix}")
if not _has_gate_fix:
    raise SystemExit(
        "KHÔNG tìm thấy fix gate collapse trong corrector_model.py vừa clone — GIT_BRANCH "
        f"'{GIT_BRANCH}' đang trỏ tới commit CŨ (trước khi fix). Kiểm tra lại biến GIT_BRANCH "
        "ở đầu cell này, hoặc branch trên GitHub đã bị đổi/xoá."
    )


## Bước 4 — Tải checkpoint Model 1 + dataset (phase1, chứa `hcm0031`) từ Google Drive

**Đã điền sẵn cả 2 link** (checkpoint bạn vừa cung cấp + dataset dùng lại đúng link đã
dùng ở `Round1/bts-digital-twin-public.ipynb`) — bạn chỉ cần: mở notebook này trên
Kaggle, điền `GITHUB_TOKEN` ở Bước 3 (nếu repo Private), bật GPU, **Run All**.

Lưu ý: `DATASET_DRIVE_LINK` là file zip của **CẢ thư mục `phase1`** (13 scene: 5
`public_set` + 8 `private_set1`, đúng như `bts-digital-twin-public.ipynb` Bước 4 dùng) —
không phải zip riêng `hcm0031`. Cell dưới tự lọc đúng scene `SCENE_NAME` bên trong zip
đó (không lấy nhầm 1 trong 12 scene còn lại).

In [ ]:
SCENE_NAME = "hcm0031"

CHECKPOINT_DRIVE_LINK = "https://drive.google.com/file/d/1d8r9T8_ohh6LGK8LCr3D1nadf8neljNw/view?usp=drive_link"
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1GUflBBz4hrVtkMcHLfLMwUYk4HsJkXFu/view?usp=drive_link"

assert CHECKPOINT_DRIVE_LINK, "Chưa điền CHECKPOINT_DRIVE_LINK."
assert DATASET_DRIVE_LINK, "Chưa điền DATASET_DRIVE_LINK."

import os
from pathlib import Path

os.makedirs("/kaggle/working/_ckpt_raw", exist_ok=True)
!gdown "{CHECKPOINT_DRIVE_LINK}" -O /kaggle/working/gs_model.zip
!unzip -q -o /kaggle/working/gs_model.zip -d /kaggle/working/_ckpt_raw

os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown "{DATASET_DRIVE_LINK}" -O /kaggle/working/phase1.zip
!unzip -q -o /kaggle/working/phase1.zip -d /kaggle/working/_dataset_raw

# Tự dò thư mục chứa "cfg_args" (gốc gs_model/) — không giả định cứng độ sâu sau giải nén.
ckpt_candidates = [p.parent for p in Path("/kaggle/working/_ckpt_raw").rglob("cfg_args")
                   if "__MACOSX" not in p.parts]
assert ckpt_candidates, "Không tìm thấy 'cfg_args' trong zip checkpoint — kiểm tra lại đã nén đúng thư mục gs_model/ chưa."
MODEL_DIR = ckpt_candidates[0]

# DATASET_DRIVE_LINK chứa CẢ thư mục phase1 (13 scene) — PHẢI lọc đúng đích danh
# SCENE_NAME (tên thư mục cha của "test_poses.csv" == SCENE_NAME), KHÔNG lấy đại
# candidate[0] (dễ trúng nhầm 1 trong 12 scene khác nếu chỉ tìm theo "test_poses.csv").
scene_candidates = [p.parent.parent for p in Path("/kaggle/working/_dataset_raw").rglob("test_poses.csv")
                    if "__MACOSX" not in p.parts and p.parent.parent.name == SCENE_NAME]
assert scene_candidates, (
    f"Không tìm thấy scene '{SCENE_NAME}' (thư mục tên đúng, chứa test/test_poses.csv) trong "
    f"zip dataset vừa giải nén — kiểm tra lại DATASET_DRIVE_LINK có đúng là zip phase1 không."
)
SCENE_ROOT = scene_candidates[0]
TEST_POSES_CSV = SCENE_ROOT / "test" / "test_poses.csv"
TEST_IMAGES_DIR = SCENE_ROOT / "test" / "images"
TRAIN_IMAGES_DIR = SCENE_ROOT / "train" / "images"
SPARSE_DIR = SCENE_ROOT / "train" / "sparse" / "0"

for _p in (TEST_POSES_CSV, TEST_IMAGES_DIR, TRAIN_IMAGES_DIR, SPARSE_DIR):
    assert _p.exists(), f"Thiếu {_p} — kiểm tra lại zip dataset có đủ train/+test/ cho scene {SCENE_NAME} không."

print("MODEL_DIR       =", MODEL_DIR)
print("SCENE_ROOT       =", SCENE_ROOT)
print("TEST_POSES_CSV   =", TEST_POSES_CSV, "|", len(list(open(TEST_POSES_CSV))) - 1, "pose")
print("TEST_IMAGES_DIR  =", TEST_IMAGES_DIR, "|", len(list(TEST_IMAGES_DIR.glob("*"))), "ảnh")
print("TRAIN_IMAGES_DIR =", TRAIN_IMAGES_DIR, "|", len(list(TRAIN_IMAGES_DIR.glob("*"))), "ảnh")
print("SPARSE_DIR       =", SPARSE_DIR)


## Bước 5 — Nạp Model 1 + render toàn bộ pose TEST (TRƯỚC khi sửa)

Import `08_build_corrector_dataset.py` làm THƯ VIỆN (không chạy `main()`) để tái dùng
`build_minicam()`/`read_cfg_args()`/`read_pipeline_train_flags()`/`load_train_poses()`
— đúng logic đã test ở `tests/test_corrector_pipeline.py`, không viết lại.


In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
from PIL import Image as PILImage

sys.path.insert(0, "/kaggle/working/pipeline")
from common.poses import read_test_poses, assert_centered_principal_point

def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod08 = load_module("/kaggle/working/pipeline/scripts/08_build_corrector_dataset.py", "mod08")

cfg = mod08.read_cfg_args(MODEL_DIR)
train_flags = mod08.read_pipeline_train_flags(MODEL_DIR)  # checkpoint cũ -> rỗng, in cảnh báo là BÌNH THƯỜNG
sh_degree = cfg.get("sh_degree", 3)
antialiasing = bool(train_flags.get("antialiasing", False))
print(f"sh_degree={sh_degree}  antialiasing={antialiasing} (checkpoint cũ, xác nhận qua notebook train "
      f"gốc KHÔNG bật antialiasing -> mặc định False ở đây là ĐÚNG, không phải đoán mù)")

def find_checkpoint_ply(model_dir: Path):
    """Tìm iteration_<N>/point_cloud.ply LỚN NHẤT trong model_dir — chấp nhận CẢ 2 layout:
    model_dir/point_cloud/iteration_<N>/point_cloud.ply (chuẩn gaussian-splatting, đúng
    layout 02_train_baseline.sh/06_train_refine.sh tạo ra) LẪN
    model_dir/iteration_<N>/point_cloud.ply (thiếu 1 cấp "point_cloud/" — hay gặp khi tải
    tay từ tab Output của Kaggle rồi lỡ chọn nhầm cấp thư mục lúc download, như trường hợp
    checkpoint hcm0031 này). KHÔNG dùng mod08.find_latest_iteration() (chỉ chấp nhận layout
    đầu) để tránh phải bắt người dùng nén lại zip đúng layout chuẩn."""
    candidates = []
    for p in model_dir.rglob("point_cloud.ply"):
        parent = p.parent
        if not parent.name.startswith("iteration_"):
            continue
        try:
            n = int(parent.name.split("_")[-1])
        except ValueError:
            continue
        candidates.append((n, p))
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy 'iteration_<N>/point_cloud.ply' nào trong {model_dir} (đã thử "
            f"cả layout có/không có lớp con 'point_cloud/') — kiểm tra lại zip checkpoint."
        )
    candidates.sort(key=lambda t: t[0])
    iteration, ply_path = candidates[-1]
    return iteration, ply_path


iteration, ply_path = find_checkpoint_ply(MODEL_DIR)
print(f"Checkpoint: {ply_path} (iteration {iteration})")

import torch
gaussians = mod08.GaussianModel(sh_degree)
gaussians.load_ply(str(ply_path))
print(f"Đã nạp {gaussians.get_xyz.shape[0]:,} Gaussian.")

background = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")
pipe = mod08._PipelineParamsStub()
pipe.antialiasing = antialiasing

def render_pose_u8(pose):
    cam = mod08.build_minicam(pose)
    with torch.no_grad():
        out = mod08.render(cam, gaussians, pipe, background)
    img = out["render"].clamp(0, 1).detach().cpu().numpy().transpose(1, 2, 0)
    return (img * 255.0).round().astype(np.uint8)

BEFORE_DIR = Path("/kaggle/working/eval/test_renders_before")
BEFORE_DIR.mkdir(parents=True, exist_ok=True)

test_poses = read_test_poses(TEST_POSES_CSV)
print(f"===== Render {len(test_poses)} pose TEST (TRƯỚC khi sửa) =====")
for i, pose in enumerate(test_poses):
    assert_centered_principal_point(pose)
    img = render_pose_u8(pose)
    stem = Path(pose.image_name).stem
    PILImage.fromarray(img).save(BEFORE_DIR / f"{stem}.png")
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(test_poses)}]")
print(f"-> Xong, {len(test_poses)} ảnh tại {BEFORE_DIR}")


## Bước 6 — Tính Score THẬT (TRƯỚC khi sửa)

So ảnh render Bước 5 với ảnh GT test THẬT (`test/images/`) — dùng ĐÚNG công thức Score
của BTC, import trực tiếp từ `04_eval_metrics.py` (không viết lại công thức).


In [ ]:
from types import SimpleNamespace

mod04 = load_module("/kaggle/working/pipeline/scripts/04_eval_metrics.py", "mod04")

lpips_fn = None
if mod04._HAS_LPIPS:
    import lpips as _lpips
    lpips_fn = _lpips.LPIPS(net="alex").cuda()
else:
    print("[CẢNH BÁO] Chưa cài package lpips.")

dummy_scene = SimpleNamespace(name="hcm0031")
rows_before = mod04.eval_scene(dummy_scene, BEFORE_DIR, TEST_IMAGES_DIR, lpips_fn)
score_before = mod04.print_stats("hcm0031 — TRƯỚC khi sửa (Model 1 gốc)", rows_before, psnr_max=50.0)


## Bước 7 — Render toàn bộ pose TRAIN → cặp (render, GT) cho Model 2

Dùng ĐÚNG checkpoint Model 1 vừa nạp, render lại pose TRAIN — dữ liệu để Model 2 học sửa
lỗi.

**Bug thật đã tìm khi chạy lần đầu:** `train/sparse/0/` trong zip dataset gốc BTC cấp là
sparse **CHƯA undistort** (camera model `SIMPLE_RADIAL`, có méo ống kính nhẹ) —
`load_train_poses()` chỉ đọc được `SIMPLE_PINHOLE`/`PINHOLE` (`ValueError: camera model
'SIMPLE_RADIAL' không được hỗ trợ`). `test_poses.csv` KHÔNG bị lỗi này (đã tự chứa sẵn
fx/fy/cx/cy dạng pinhole, không tham chiếu camera model COLMAP) — lỗi CHỈ xảy ra ở bước
này (pose TRAIN, đọc trực tiếp từ `.bin`). Đúng quy trình chuẩn mà `cfg_args` của chính
checkpoint này xác nhận đã dùng lúc train (`source_path` trỏ vào `colmap/dense`, tức bản
ĐÃ undistort) — phải tự undistort lại y hệt trước khi render, dùng
`common.colmap_runner.use_provided_sparse()` (đúng hàm `01_run_colmap.py` dùng cho 7
scene Round 2, port nguyên vẹn ở đây, không viết lại).


**Nghi vấn kiến trúc cần TỰ kiểm chứng bằng số liệu (không suy đoán):** Model 1 (3DGS)
được train TRỰC TIẾP để tối thiểu hoá lỗi ĐÚNG TRÊN các pose TRAIN này (đó là chính mục
tiêu tối ưu của `train.py`) — nên PSNR/SSIM giữa render và GT tại pose TRAIN thường CAO
HƠN NHIỀU so với PSNR/SSIM tại pose TEST thật (Bước 6 đo được, pose Model 1 CHƯA từng
tối ưu tới). Nếu đúng vậy, cặp (render, GT) ở Bước 7 là bài toán "sửa lỗi RẤT NHỎ" — Model
2 học trên tín hiệu quá dễ, có thể không đủ để sửa lỗi LỚN HƠN thật sự tồn tại ở pose
test — bất kể cơ chế gate/residual có đúng hay không. Cell dưới tự đo PSNR/SSIM TRAIN
ngay sau khi render xong, in cạnh Score TEST (Bước 6) để so sánh trực tiếp — đây là bằng
chứng số liệu, không phải phỏng đoán.

In [ ]:
from common.colmap_runner import use_provided_sparse

UNDISTORT_WORKDIR = Path("/kaggle/working/colmap_undistort")
undistort_result = use_provided_sparse(
    images_dir=TRAIN_IMAGES_DIR, sparse_dir=SPARSE_DIR, workdir=UNDISTORT_WORKDIR,
)
UNDISTORTED_SPARSE_DIR = undistort_result["dense_dir"] / "sparse" / "0"
UNDISTORTED_IMAGES_DIR = undistort_result["dense_dir"] / "images"
print(f"Undistort xong: {undistort_result['num_reg_images']} ảnh, "
      f"{undistort_result['num_points3D']} điểm, thiếu {len(undistort_result['missing_images'])} ảnh "
      f"(camera model gốc SIMPLE_RADIAL -> PINHOLE sạch, log: {undistort_result['log_path']})")

CORRECTOR_DATASET_DIR = Path("/kaggle/working/corrector_dataset")
render_dir = CORRECTOR_DATASET_DIR / "render"
gt_dir = CORRECTOR_DATASET_DIR / "gt"
render_dir.mkdir(parents=True, exist_ok=True)
gt_dir.mkdir(parents=True, exist_ok=True)

train_poses = mod08.load_train_poses(UNDISTORTED_SPARSE_DIR)
names_sorted = sorted(train_poses.keys())
print(f"===== Render {len(names_sorted)} pose TRAIN (cho Model 2) =====")
n_saved = 0
for i, name in enumerate(names_sorted):
    gt_path = UNDISTORTED_IMAGES_DIR / name
    if not gt_path.exists():
        print(f"  [BỎ QUA] {name}: không thấy {gt_path}")
        continue
    pose = train_poses[name]
    assert_centered_principal_point(pose)
    img = render_pose_u8(pose)
    stem = Path(name).stem
    PILImage.fromarray(img).save(render_dir / f"{stem}.png")
    with PILImage.open(gt_path) as gt_im:
        gt_im.convert("RGB").save(gt_dir / f"{stem}.png", format="PNG")
    n_saved += 1
    if n_saved % 20 == 0:
        print(f"  [{n_saved}/{len(names_sorted)}]")

import json as _json
(CORRECTOR_DATASET_DIR / "manifest.json").write_text(_json.dumps(
    {"scene": "hcm0031", "source_iteration": iteration, "n_pairs": n_saved}))
print(f"-> Xong, {n_saved} cặp (render, GT) tại {CORRECTOR_DATASET_DIR}")

# ĐO NGAY: PSNR/SSIM giữa render và GT tại pose TRAIN — so trực tiếp với Score TEST
# (Bước 6) để kiểm chứng nghi vấn kiến trúc ở markdown phía trên bằng SỐ LIỆU THẬT.
rows_train = mod04.eval_scene(dummy_scene, render_dir, gt_dir, lpips_fn)
score_train = mod04.print_stats("hcm0031 — render TRAIN vs GT (dữ liệu Model 2 học)", rows_train, psnr_max=50.0)
print()
print("=" * 70)
print(f"  PSNR trung bình pose TRAIN (Model 2 học trên đây) vs pose TEST (Bước 6):")
_psnr_train = float(np.mean([r[1] for r in rows_train]))
_psnr_test = float(np.mean([r[1] for r in rows_before]))
print(f"  PSNR train={_psnr_train:.2f}  PSNR test={_psnr_test:.2f}  chênh={_psnr_train - _psnr_test:+.2f} dB")
if _psnr_train - _psnr_test > 3.0:
    print("  [CẢNH BÁO] Chênh lệch > 3dB — xác nhận nghi vấn: dữ liệu train Model 2 học "
          "là bài toán DỄ HƠN NHIỀU so với lỗi thật cần sửa ở pose test. Model 2 có thể "
          "học ra correction quá YẾU/bảo thủ để tạo khác biệt thật ở Bước 9. Cân nhắc: "
          "giảm --gate_sparsity_weight (cho phép sửa lan rộng hơn), tăng STEPS, hoặc "
          "(giải pháp gốc rễ) train Model 1 bớt lại vài ảnh KHÔNG đưa vào để dành riêng "
          "làm dữ liệu 'giống test thật' cho Model 2 học — đánh đổi với lựa chọn ban đầu "
          "'train 100% dữ liệu, không giữ holdout' đã chọn.")
print("=" * 70)


## Bước 8 — Train Model 2 (mạng sửa lỗi pixel, tự học vùng cần sửa)

Import `09_train_corrector.py` làm thư viện để tái dùng `CorrectorPatchDataset` +
`ResidualCorrectorNet` + `save_checkpoint`/`load_checkpoint` (đúng code đã test) — **giờ
đã có đúng ngữ nghĩa `--resume`/`--overwrite` như bản CLI thật** (trước đây chạy lại cell
này = train lại từ đầu, KHÔNG phải tinh chỉnh tiếp — đã sửa theo góp ý review):

- `RESUME=False, OVERWRITE=False` (mặc định) — train lần đầu; nếu đã có checkpoint sẵn,
  DỪNG LẠI báo lỗi rõ (không âm thầm ghi đè hay âm thầm train lại từ đầu).
- `RESUME=True` — nạp checkpoint hiện có, train tiếp thêm đúng `STEPS` bước (số bước
  LUỸ KẾ lưu trong field `"step"` của checkpoint, không cần đổi tên thư mục gì).
- `OVERWRITE=True` — xoá checkpoint hiện có, train lại từ đầu (CỐ Ý).

Cũng thêm **tập validation nhỏ riêng của Model 2** (`VAL_FRACTION`, mặc định 10% số cặp
render/GT TRAIN) — KHÔNG liên quan gì tới việc Model 1 (3DGS) có giữ holdout hay không
(đó là quyết định đã chốt: train 100%, không đổi). Đây chỉ là tách 1 phần nhỏ trong
CHÍNH 200 cặp (render, GT) mà Model 2 nhìn thấy, KHÔNG dùng để train patch (val loss đo
trên đó để biết corrector có overfit đúng các cặp đã học hay không) — tín hiệu tune thật
đầu tiên, dù vẫn cùng "miền dễ" (pose train) như nghi vấn kiến trúc đã nêu ở Bước 7.


In [ ]:
STEPS = 4000
RESUME = False        # <-- True = train TIẾP checkpoint đã có (không train lại từ đầu)
OVERWRITE = False     # <-- True = xoá checkpoint cũ, train lại từ đầu (CỐ Ý)
LOSS = "l1"            # "l1" hoặc "l1_ssim" (SSIM giúp giữ cấu trúc, thử nếu ảnh "sau" bị nhoè)
SSIM_WEIGHT = 0.2       # chỉ dùng nếu LOSS="l1_ssim"
GATE_SPARSITY_WEIGHT = 0.01   # xem Bước 7: tăng nếu gate "sửa tràn lan", giảm nếu gate quá rụt rè
VAL_FRACTION = 0.10     # tỉ lệ cặp render/GT giữ lại làm val riêng của Model 2 (xem markdown trên)

if RESUME and OVERWRITE:
    raise SystemExit("RESUME và OVERWRITE loại trừ nhau — chỉ chọn 1.")

mod09 = load_module("/kaggle/working/pipeline/scripts/09_train_corrector.py", "mod09")
mod_cm = load_module("/kaggle/working/pipeline/common/corrector_model.py", "corrector_model")

CORRECTOR_MODEL_DIR = Path("/kaggle/working/corrector_model")
CKPT_PATH = CORRECTOR_MODEL_DIR / "corrector.pt"

if CKPT_PATH.exists():
    if OVERWRITE:
        CKPT_PATH.unlink()
        print(f"[dọn dẹp] Đã xoá checkpoint cũ {CKPT_PATH} — train lại từ đầu.")
    elif not RESUME:
        raise SystemExit(
            f"Đã có checkpoint tại {CKPT_PATH} — đặt RESUME=True để train tiếp (thêm STEPS "
            f"bước nữa), hoặc OVERWRITE=True để huỷ và train lại từ đầu."
        )
elif RESUME:
    raise SystemExit(f"RESUME=True nhưng không thấy checkpoint nào tại {CKPT_PATH} — đặt RESUME=False để train lần đầu.")

# Tách val riêng của Model 2 (xem markdown Bước 8) — deterministic theo seed, không đổi
# giữa các lần chạy lại (miễn CORRECTOR_DATASET_DIR không đổi).
_all_stems = sorted(p.stem for p in (CORRECTOR_DATASET_DIR / "render").glob("*.png"))
_rng_split = __import__("random").Random(123)
_shuffled = _all_stems.copy()
_rng_split.shuffle(_shuffled)
_n_val = max(1, int(len(_shuffled) * VAL_FRACTION))
val_stems = sorted(_shuffled[:_n_val])
train_stems = sorted(_shuffled[_n_val:])
print(f"Tách dữ liệu Model 2: {len(train_stems)} train / {len(val_stems)} val (VAL_FRACTION={VAL_FRACTION})")

torch.manual_seed(42)
model = mod_cm.build_corrector(num_blocks=6, channels=64).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

step_start = 0
loss_history = []
if RESUME:
    payload = mod_cm.load_checkpoint(CKPT_PATH, model, optimizer, map_location="cuda")
    if payload.get("dataset_source_iteration") != iteration:
        raise SystemExit(
            f"Checkpoint train trên dataset ứng iteration={payload.get('dataset_source_iteration')} "
            f"nhưng corrector_dataset hiện tại ứng iteration={iteration} — không khớp, KHÔNG resume."
        )
    step_start = payload["step"]
    loss_history = payload.get("loss_history", [])
    print(f"Resume từ checkpoint step={step_start}.")

train_dataset = mod09.CorrectorPatchDataset(CORRECTOR_DATASET_DIR, patch_size=256, seed=42, stems=train_stems)
val_dataset = mod09.CorrectorPatchDataset(CORRECTOR_DATASET_DIR, patch_size=256, seed=999, stems=val_stems)
loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, drop_last=True)
batches = mod09._endless_batches(loader)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, drop_last=False)

l1_loss_fn = torch.nn.L1Loss()
model.train()
target_step = step_start + STEPS
print(f"===== Train Model 2 — step {step_start} -> {target_step} (loss={LOSS}) =====")
for step in range(step_start + 1, target_step + 1):
    render_batch, gt_batch = next(batches)
    render_batch, gt_batch = render_batch.cuda(), gt_batch.cuda()

    output, gate, _residual = model.forward_with_gate(render_batch)
    l1 = l1_loss_fn(output, gt_batch)
    if LOSS == "l1_ssim":
        ssim_val = mod_cm.ssim(output, gt_batch)
        recon_loss = (1.0 - SSIM_WEIGHT) * l1 + SSIM_WEIGHT * (1.0 - ssim_val)
    else:
        recon_loss = l1
    gate_mean = gate.mean()
    loss = recon_loss + GATE_SPARSITY_WEIGHT * gate_mean

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0 or step == target_step:
        model.eval()
        with torch.no_grad():
            val_l1s = []
            for vr, vg in val_loader:
                vr, vg = vr.cuda(), vg.cuda()
                vout = model(vr)
                val_l1s.append(l1_loss_fn(vout, vg).item())
        val_l1 = sum(val_l1s) / len(val_l1s)
        model.train()
        entry = {"step": step, "loss": float(loss.item()), "l1": float(l1.item()),
                 "gate_mean": float(gate_mean.item()), "val_l1": val_l1}
        loss_history.append(entry)
        print(f"  [{step}/{target_step}] loss={loss.item():.5f} l1={l1.item():.5f} "
              f"gate_mean={gate_mean.item():.4f} val_l1={val_l1:.5f}")

mod_cm.save_checkpoint(
    CKPT_PATH, model, optimizer, step=target_step,
    train_args={"patch_size": 256, "batch_size": 8, "lr": 2e-4, "loss": LOSS, "ssim_weight": SSIM_WEIGHT,
                "gate_sparsity_weight": GATE_SPARSITY_WEIGHT, "val_fraction": VAL_FRACTION},
    dataset_source_iteration=iteration, loss_history=loss_history,
)
print(f"-> Xong. Checkpoint Model 2: {CKPT_PATH} (step={target_step})")
print(f"-> Muốn train tiếp: đặt RESUME=True ở đầu cell này rồi chạy lại (KHÔNG cần đổi gì khác).")


## Bước 9 — Áp Model 2 lên render TEST → tính Score SAU khi sửa

Import `10_apply_corrector.py` làm thư viện để tái dùng `apply_tiled()` (tiled inference
+ trộn biên + gate heatmap) — đúng logic đã test.


In [ ]:
mod10 = load_module("/kaggle/working/pipeline/scripts/10_apply_corrector.py", "mod10")

AFTER_DIR = Path("/kaggle/working/eval/test_renders_after")
GATE_DIR = Path("/kaggle/working/eval/test_gate_maps")
AFTER_DIR.mkdir(parents=True, exist_ok=True)
GATE_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
print(f"===== Áp Model 2 lên {len(test_poses)} render TEST =====")
gate_means = []
for i, pose in enumerate(test_poses):
    stem = Path(pose.image_name).stem
    with PILImage.open(BEFORE_DIR / f"{stem}.png") as im:
        img01 = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
    corrected, gate_map = mod10.apply_tiled(model, img01, tile_size=512, overlap=32,
                                             device="cuda", return_gate=True)
    PILImage.fromarray((corrected * 255.0).round().astype(np.uint8)).save(AFTER_DIR / f"{stem}.png")
    # KHÔNG truyền mode="L" — Pillow tự suy ra đúng từ mảng 2D uint8, tham số này sẽ bị
    # xoá hẳn ở Pillow 13 (2026-10-15, DeprecationWarning đã thấy khi chạy thật).
    PILImage.fromarray((gate_map[:, :, 0] * 255.0).round().astype(np.uint8)).save(GATE_DIR / f"{stem}.png")
    gate_means.append(float(gate_map.mean()))
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(test_poses)}]")
print(f"-> Xong. gate_mean trung bình toàn bộ test: {sum(gate_means)/len(gate_means):.4f}")

rows_after = mod04.eval_scene(dummy_scene, AFTER_DIR, TEST_IMAGES_DIR, lpips_fn)
score_after = mod04.print_stats("hcm0031 — SAU khi sửa (Model 1 + Model 2)", rows_after, psnr_max=50.0)


## Bước 10 — So sánh TRƯỚC/SAU + xem ảnh mẫu

Đây là câu trả lời trực tiếp cho câu hỏi "Model 2 giúp tăng bao nhiêu điểm" — số liệu
THẬT, không phải suy đoán.


In [ ]:
delta = score_after - score_before
print("=" * 70)
print(f"  Score TRƯỚC (Model 1 gốc)     : {score_before:.4f}")
print(f"  Score SAU (Model 1 + Model 2) : {score_after:.4f}")
print(f"  Chênh lệch                    : {delta:+.4f}  ({'TĂNG' if delta > 0 else 'GIẢM' if delta < 0 else 'không đổi'})")
print("=" * 70)
print("(công thức Score = 0.4*(1-LPIPS) + 0.3*SSIM + 0.3*PSNR_norm, PSNR_max=50.0 ước lượng "
      "tham khảo — xem docstring 04_eval_metrics.py)")


In [ ]:
from IPython.display import display

sample_names = sorted(p.name for p in BEFORE_DIR.glob("*.png"))[:3]
for name in sample_names:
    stem = Path(name).stem
    gt_path = TEST_IMAGES_DIR / f"{stem}.JPG"
    if not gt_path.exists():
        gt_candidates = list(TEST_IMAGES_DIR.glob(f"{stem}.*"))
        gt_path = gt_candidates[0] if gt_candidates else None

    im_before = PILImage.open(BEFORE_DIR / name).convert("RGB")
    im_gate = PILImage.open(GATE_DIR / name).convert("L").convert("RGB")
    im_after = PILImage.open(AFTER_DIR / name).convert("RGB")
    w, h = im_before.size
    panels = [im_before, im_gate, im_after]
    labels = "trước | gate heatmap | sau"
    if gt_path and gt_path.exists():
        im_gt = PILImage.open(gt_path).convert("RGB").resize((w, h))
        panels.append(im_gt)
        labels += " | GT thật"

    combo = PILImage.new("RGB", (w * len(panels) + 10 * (len(panels) - 1), h), (255, 255, 255))
    x = 0
    for p in panels:
        combo.paste(p, (x, 0))
        x += w + 10
    print(f"--- {name} ({labels}) ---")
    display(combo.resize((min(1600, combo.width), int(combo.height * min(1600, combo.width) / combo.width))))
